In [1]:
import torch
from datetime import datetime
from einops import repeat
import numpy as np
import torch.nn as nn
import torch.optim as optim
from transformers import get_cosine_schedule_with_warmup
import os
from torch.utils.data import Dataset, DataLoader
from NKD_models import *

from dataset_loaders import Train_img_Dataset, TrainingDataset
# from models.NKD_linear_layer import NKD_latent_layers, NKD_latent_layers_v2, save_checkpoint, load_checkpoint
# from NKD_kenel_layer import NKD_kernel_layers_v1
# from NKD_test import Test_denoisers


/opt/anaconda3/envs/nkd-aditya/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def TestDenoisers(data_loader, denoisers, device, out_folder, p=0, state='', verbose=False):
    if verbose:
        print('==== Test ====')
    # Create a figure to save the images
    grid = (len(denoisers), 3)

    # Loop through the validation loader
    for i, data in enumerate(data_loader):
        if verbose:
            print(f'Image:- {i}')

        clean_image, noisy_image, sigma = data  # dataset returns a tuple of (noisy_image, clean_image)
        
        # Move images to the same device as the model
        noisy_image = noisy_image.to(device=device, dtype=torch.float32).squeeze(0)
        unpadded_noisy_image = noisy_image[:, p:-p, p:-p]
        clean_image = clean_image.to(device=device, dtype=torch.float32).squeeze(0)[:, p:-p, p:-p]
        sigma = sigma.to(device=device, dtype=torch.float32)

        psnr_noisy = calculate_psnr(clean_image, unpadded_noisy_image)
        ssim_noisy = calculate_ssim(clean_image, unpadded_noisy_image)
        if verbose:
            print(f'Noisy: psnr = {psnr_noisy:.2f} | ssim = {ssim_noisy:.4f}')
        titles = []
        images = []

        for j, name in enumerate(denoisers.keys()):
            # Pass the noisy image through the model to get the denoised image
            if name == 'BM3D':
                with torch.no_grad():
                    denoised_image = denoisers[name](unpadded_noisy_image, sigma.squeeze().item())
                unpadded_noisy_image = unpadded_noisy_image.to(device)
                denoised_image = denoised_image.to(device)

            elif 'nkd' in name.lower():
                with torch.no_grad():
                    denoised_image = denoisers[name].forward_img(noisy_image, sigma)
                    if verbose:
                        print(denoised_image.shape, clean_image.shape)
            else:
                with torch.no_grad():
                    denoised_image = denoisers[name](unpadded_noisy_image, sigma)
            
            # Calculate PSNR of denoised image
            psnr = calculate_psnr(clean_image, denoised_image)
            ssim = calculate_ssim(clean_image, denoised_image)
            if verbose:
                print(f'{name}: psnr = {psnr:.2f} | ssim = {ssim:.4f}')
            
            images.append(unpadded_noisy_image)
            titles.append(f'Noisy Image({psnr_noisy:.2f}dB)')

            images.append(denoised_image)
            titles.append(f'{name}: Denoised Image({psnr:.2f}dB)')

            images.append(clean_image)
            titles.append(f'Clean Image')

        show_tensor_image(images, grid, titles, save=True, save_path=os.path.join(out_folder, f'saved_test_{i}_{state}.png'))


In [3]:
# val=0

In [4]:
# device = 'cuda:0'
# class config:
#             def __init__(self):
#                 pass
# a = NKD_denoisers.load_checkpoint('/home/aditya/NKD/output__concat_kl_5101512_22_130_30/ckpt/model_step_12000.pth',
#                                   device=device)

# a1 = NKD_denoisers.load_checkpoint('/home/aditya/NKD/output_diff_kl_5101512_22_130_30/ckpt/model_step_12000.pth',
#                                   device=device)

# a2 = NKD_denoisers.load_checkpoint('/home/aditya/NKD/output_rbf_510151210_22_130_30/ckpt/model_step_12000.pth',
#                                   device=device)

# padding_dim = a[0].window_rad+a[0].patch_rad
# testset = Train_img_Dataset('./data/img_train', 'gray', 30, 30, 
#                         patch_size=2*(padding_dim)+100, 
#                         imgs_per_epoch=1, req_patches=False)

# test_loader = DataLoader(
#     testset,
#     batch_size=1,  # Process one full image at a time
#     shuffle=False,
#     pin_memory=True
# )

# TestDenoisers(
#         test_loader, {
#             'NKD kernel function learn: \n concatenate patchs \n': a[0],
#             'NKD kernel function learn: \n diff. sq. of patch features \n': a1[0],
#             'NKD RBG feature learning\n': a2[0],
#         }, device, out_folder='./', 
#         p=padding_dim, state=val,
#     )
# val+=1

In [5]:
def train(args, verbose = False):
    # Setup device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    trainset = TrainingDataset('./data/train', args.color_mode, args.sigma_min, args.sigma_max, 
                                 patch_size=args.patch_size, imgs_per_epoch=args.imgs_per_epoch,
                                 req_patches=args.req_patches, no_patches=args.no_patches)

    train_loader = DataLoader(
                trainset,
                batch_size=1,  # Process one full image at a time
                shuffle=False,
                pin_memory=True
            )
    
    # padding_dim = args.window_rad+args.patch_rad
    # testset = Train_img_Dataset('./data/img_train', args.color_mode, args.sigma_max, args.sigma_max, 
    #                              patch_size=2*(padding_dim)+args.test_imgsize, size=None,
    #                              train_subdirs=['img_baboon'],
    #                              imgs_per_epoch=1, req_patches=False)

    # test_loader = DataLoader(
    #     testset,
    #     batch_size=1,  # Process one full image at a time
    #     shuffle=False,
    #     pin_memory=True
    # )
    
    # Create model
    in_nc = out_nc = 1 if args.color_mode == 'gray' else 3
    model = architectures[args.architecture](args, device).to(device)

    # Get the current time
    time_stamp = datetime.now().strftime("%m%d%H%M%S")
    _state = f'_{str(model)}{args.sigma_min}_{args.sigma_max}_{time_stamp}'
    args.out_folder = args.out_folder+_state
    # Check if the output folder exists
    if not os.path.exists(args.out_folder):
        # Create the folder
        os.makedirs(args.out_folder)
        os.makedirs(os.path.join(args.out_folder, 'ckpt'))
        os.makedirs(os.path.join(args.out_folder, 'imgs'))
        if verbose:
            print(f"Folder created at: {args.out_folder}")

    # Setup optimizer, scheduler and criterion
    optimizer = optim.AdamW(model.parameters(), lr=args.lr)
    if args.sch == 'multistep':
        if isinstance(args.lr_step, str):
            milestones = [int(x) for x in args.lr_step.split(',')]
        else:
            milestones = [args.lr_step]
        # Multi-step learning rate scheduler
        scheduler = optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=milestones, gamma=args.lr_gamma
        )
    elif args.sch == 'cosine':
        # Cosine annealing learning rate scheduler
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=args.epochs
        )
    elif args.sch == 'warmup_cosine':
        scheduler = get_cosine_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=args.num_warmup_steps,
            num_training_steps=args.epochs,
            num_cycles=args.num_cycles
        )
    elif args.sch == 'const':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=args.epochs
        )
    else:
        raise ValueError(f"scehduler(={args.sch}) is not defined")
    criterion = nn.MSELoss()
    # For tracking best validation results
    best_psnr = 0
    best_ssim = 0

    # Training loop
    total_steps = 0
    model.train()

    losses = []
    val_metric = []

    # try:
    if args.resume_training:
        model, optimizer, scheduler, args = NKD_denoisers.load_checkpoint(args.checkpoint_path, device, optimizer, scheduler)

    model.train()
    for epoch in range(args.epochs):
        batch_count = 0
        current_lr = optimizer.param_groups[0]['lr']

        loss = model.train_step(train_loader, criterion, optimizer, lamda=args.l1_reg_lamda)
        total_steps += 1
        batch_count += 1
        losses.append(loss)

        # Print progress
        if total_steps % args.progress_freq == 0:
            print(f"Epoch {epoch+1}/{args.epochs} | "
                    f"Loss: {loss:.6f} | lr = {current_lr}", end='\r')

        # Regular checkpoint saving
        if total_steps % args.save_freq == 0:
            if verbose:
                print('saving checkpoint')
            model.save_checkpoint(args.out_folder, {
                    'step': total_steps,
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'scheduler_state_dict':scheduler.state_dict(),
                    'loss': loss,
                    'args': args,
                }, total_steps)

        # Save epoch loss to log
        with open(os.path.join(args.out_folder, 'training_log.txt'), 'a') as f:
            f.write(f'{epoch+1},{loss:.6f},{current_lr}\n')

        if args.sch!='const':
            scheduler.step()

        # # Save Image
        # if args.save_img and total_steps % args.save_img_freq == 0:
        #     TestDenoisers(test_loader, {'NKD : code' : model}, device, 
        #                   out_folder=os.path.join(args.out_folder, 'imgs'), 
        #                   p=padding_dim, state=f'{total_steps}', verbose=False)

    # except KeyboardInterrupt:
    #     print("\nTraining interrupted. Saving checkpoint...")

    # except Exception as e:
    #     print(f"\nERROR: {str(e)}")
    #     raise e

    # finally:
    print("\nTraining complete!")
    print(f"Best PSNR: {best_psnr:.2f}dB")
    print(f"Best SSIM: {best_ssim:.4f}")

    # Clean up
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, losses, val_metric


In [6]:
if __name__ == "__main__":
    # args.seed, args.in_folder, args.out_folder, 
    # args.color_mode:  'gray' or 'rgb', args.sigma_min, args.sigma_max, args.patch_size, 
    # args.epochs, args.batch_size, args.lr, args.sch: 'multistep', 'warmup_cosine' or 'cosine', args.lr_step, args.lr_gamma, 
    # args.val_freq, args.save_freq, args.progress_freq, args.save_img_freq,
    # args.blind, args.patch_rad, args.window_rad, args.in_channel, args.latent_dim, args.latent_channel_dim
    seed = 100
    torch.manual_seed(seed)
    np.random.seed(seed)
    class config:
        def __init__(self):
            pass

    args = config()

    # MODEL PARAMETERS
    # architecture = ['NKD_concat_kernel_learn']
    architecture = ['NKD_diff_kernel_learn', 'NKD_concat_kernel_learn', 'NKD_rbf_feat_learn']
    patch_rads = [4]
    args.window_rad = 7
    args.in_channel = 1
    args.latent_dim = 1024
    args.latent_channel_dim = 10
    args.residual_depth = 2
    args.proj_depth = 5
    args.blind = True

    # TRAINING PARAMETERS
    args.in_folder = './data'
    args.out_folder = './output'
    args.color_mode = 'gray'
    args.sigma_min = 30
    args.sigma_max = 30
    args.req_patches = True
    args.patch_size = 128
    args.epochs = 50000
    args.batch_size = 1
    args.lr = 1e-4
    args.sch = 'const'
    args.l1_reg_lamda = 0.00
    # #for warmup_cosine
    # args.num_warmup_steps = 200
    # args.num_cycles = 0.5
    # #for multistep
    # args.lr_step = None
    # args.lr_gamma = None

    # PROGRESS DATA PARAMETERS
    args.val_freq = 100
    args.save_freq = 200
    args.save_img = True
    args.test_imgsize = 32
    args.save_img_freq = 200
    args.progress_freq = 1

    args.imgs_per_epoch = 1
    args.factor = None
    args.no_patches = 512

    # RESUME TRAINING ?
    args.resume_training = False
    args.checkpoint_path = ''
    verbose = False

    for patch_rad in patch_rads:
        for arch in architecture:
            print(arch, patch_rad)
            args.out_folder = './output'
            args.architecture = arch
            args.patch_rad = patch_rad
            if args.req_patches:
                args.patch_size = 2*(args.window_rad+args.patch_rad)+1
            else:
                pass
            _ = train(args, verbose)


NKD_diff_kernel_learn 4

Loading training images...
Loaded 400 images from BSD400


KeyboardInterrupt: 